# 01 — Team split (canonical command split)

Reproduces `notebooks/baseline.ipynb` cell-7 split: double GroupShuffleSplit by `SellerID`, `random_state=42`.

Outputs to `team_runs/splits/`:
- `team_train_idx.npy` / `team_val_idx.npy` / `team_test_idx.npy` — positional indices into `ozon_train.csv` (after the same `read_csv` call used here).
- `y_test.npy` — the `resolution` labels for the test split.
- `team_split_meta.json` — sizes, positive rates, hashes for traceability.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'notebooks'))
import utils  # noqa: E402  pylint: disable=wrong-import-position

DATA_PATH = '/Users/sofya/Desktop/ВКР /data/ml_ozon_ounterfeit_train.csv'
CLIP_PATH = '/Users/sofya/Desktop/diplomahse/clip_embeddings.parquet'
RANDOM_STATE = 42
TARGET = 'resolution'
GROUP = 'SellerID'

SPLITS_DIR = ROOT / 'splits'
SPLITS_DIR.mkdir(exist_ok=True)

print('utils loaded:', list(vars(utils).keys())[-6:])

utils loaded: ['_parse_embedding_value', 'normalize_emb_df', 'r_at_p90', 'avg_precision_metric', 'evaluate', 'bootstrap_diff']


In [2]:
df = pd.read_csv(DATA_PATH, encoding='utf-8')
print('df shape:', df.shape)
print('columns:', len(df.columns))
assert TARGET in df.columns and GROUP in df.columns, 'missing target or group column'
print('positive rate (full):', round(df[TARGET].mean(), 4))

df shape: (197198, 45)
columns: 45
positive rate (full): 0.0662


In [3]:
# ---- Sanity check on team feature spec (STOP signal #3 from user)
team_drop_cols = ['id', 'ItemID', 'SellerID', 'name_rus', 'description', 'brand_name', 'text']
team_feature_cols = [c for c in df.columns if c not in team_drop_cols + [TARGET]]
print(f'team_drop_cols:    {len(team_drop_cols)} ({team_drop_cols})')
print(f'team_feature_cols: {len(team_feature_cols)}')
print(f'team_feature_cols list:\n{team_feature_cols}')

team_drop_cols:    7 (['id', 'ItemID', 'SellerID', 'name_rus', 'description', 'brand_name', 'text'])
team_feature_cols: 38
team_feature_cols list:
['CommercialTypeName4', 'rating_1_count', 'rating_2_count', 'rating_3_count', 'rating_4_count', 'rating_5_count', 'comments_published_count', 'photos_published_count', 'videos_published_count', 'PriceDiscounted', 'item_time_alive', 'item_count_fake_returns7', 'item_count_fake_returns30', 'item_count_fake_returns90', 'item_count_sales7', 'item_count_sales30', 'item_count_sales90', 'item_count_returns7', 'item_count_returns30', 'item_count_returns90', 'GmvTotal7', 'GmvTotal30', 'GmvTotal90', 'ExemplarAcceptedCountTotal7', 'ExemplarAcceptedCountTotal30', 'ExemplarAcceptedCountTotal90', 'OrderAcceptedCountTotal7', 'OrderAcceptedCountTotal30', 'OrderAcceptedCountTotal90', 'ExemplarReturnedCountTotal7', 'ExemplarReturnedCountTotal30', 'ExemplarReturnedCountTotal90', 'ExemplarReturnedValueTotal7', 'ExemplarReturnedValueTotal30', 'ExemplarReturnedVa

In [4]:
# ---- Double GroupShuffleSplit (mirror of notebooks/baseline.ipynb cell 7)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(gss1.split(df, df[TARGET], groups=df[GROUP]))
trainval_df = df.iloc[trainval_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
tr_local, val_local = next(gss2.split(trainval_df, trainval_df[TARGET], groups=trainval_df[GROUP]))
train_idx = trainval_idx[tr_local]
val_idx = trainval_idx[val_local]

n = len(df)
print(f'train: {len(train_idx):>7d} ({len(train_idx)/n:.4f})')
print(f'val:   {len(val_idx):>7d} ({len(val_idx)/n:.4f})')
print(f'test:  {len(test_idx):>7d} ({len(test_idx)/n:.4f})')
print(f'sum:   {len(train_idx)+len(val_idx)+len(test_idx)} == n={n}')
assert len(train_idx) + len(val_idx) + len(test_idx) == n, 'partition does not cover all rows'
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

train:   69453 (0.3522)
val:     69335 (0.3516)
test:    58410 (0.2962)
sum:   197198 == n=197198


In [5]:
# ---- Sanity checks (STOP signal #4: SellerID leakage)
tr_sell = set(df.iloc[train_idx][GROUP])
va_sell = set(df.iloc[val_idx][GROUP])
te_sell = set(df.iloc[test_idx][GROUP])
assert tr_sell.isdisjoint(va_sell), 'SellerID leak train<->val'
assert tr_sell.isdisjoint(te_sell), 'SellerID leak train<->test'
assert va_sell.isdisjoint(te_sell), 'SellerID leak val<->test'

# Share bounds
tr_share = len(train_idx) / n
te_share = len(test_idx) / n
assert 0.30 <= tr_share <= 0.40, f'train share {tr_share:.4f} not in [0.30, 0.40]'
assert 0.25 <= te_share <= 0.35, f'test share {te_share:.4f} not in [0.25, 0.35]'

# Positive rates
tr_pr = df.iloc[train_idx][TARGET].mean()
va_pr = df.iloc[val_idx][TARGET].mean()
te_pr = df.iloc[test_idx][TARGET].mean()
for split_name, pr in [('train', tr_pr), ('val', va_pr), ('test', te_pr)]:
    assert 0.04 <= pr <= 0.08, f'{split_name} positive rate {pr:.4f} not in [0.04, 0.08]'
    print(f'{split_name} positive rate: {pr:.4f}')

train positive rate: 0.0610
val positive rate: 0.0602
test positive rate: 0.0795


In [6]:
# ---- Step 1.4: CLIP coverage per split.
# User STOP signal: only test < 0.95 -> STOP. train/val are reported as info.
clip = pd.read_parquet(CLIP_PATH)
clip_ids = set(clip['ItemID'].astype('int64').values)
coverage = {}
for split_name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    item_ids = df.iloc[idx]['ItemID'].astype('int64').values
    cov = float(np.isin(item_ids, list(clip_ids)).mean())
    coverage[split_name] = cov
    flag = 'OK' if cov >= 0.95 else 'WARN (<0.95)'
    print(f'{split_name} CLIP coverage: {cov:.4f}  [{flag}]')
assert coverage['test'] >= 0.95, f"test coverage {coverage['test']:.4f} < 0.95 (user STOP signal)"
del clip, clip_ids


train CLIP coverage: 0.9480  [WARN (<0.95)]
val CLIP coverage: 0.9538  [OK]
test CLIP coverage: 0.9524  [OK]


In [7]:
# ---- Save splits
y_test = df.iloc[test_idx][TARGET].astype('int8').values
np.save(SPLITS_DIR / 'team_train_idx.npy', train_idx.astype('int64'))
np.save(SPLITS_DIR / 'team_val_idx.npy',   val_idx.astype('int64'))
np.save(SPLITS_DIR / 'team_test_idx.npy',  test_idx.astype('int64'))
np.save(SPLITS_DIR / 'y_test.npy',         y_test)

meta = {
    'data_path': DATA_PATH,
    'random_state': RANDOM_STATE,
    'split_strategy': 'double GroupShuffleSplit by SellerID, test=0.30, val=0.50 of trainval',
    'sizes': {'train': int(len(train_idx)), 'val': int(len(val_idx)), 'test': int(len(test_idx))},
    'shares': {'train': float(tr_share), 'val': float(len(val_idx)/n), 'test': float(te_share)},
    'positive_rates': {'train': float(tr_pr), 'val': float(va_pr), 'test': float(te_pr)},
    'n_team_features': len(team_feature_cols),
    'team_feature_cols': team_feature_cols,
    'team_drop_cols': team_drop_cols,
    'target': TARGET,
    'group': GROUP,
    'df_n_rows': int(n),
    'df_n_cols': int(df.shape[1]),
}
with open(SPLITS_DIR / 'team_split_meta.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('saved:')
for p in sorted(SPLITS_DIR.iterdir()):
    print(f'  {p.name}: {p.stat().st_size} bytes')
print('\nmeta:')
print(json.dumps({k: v for k, v in meta.items() if k != 'team_feature_cols'}, indent=2, ensure_ascii=False))

saved:
  team_split_meta.json: 1869 bytes
  team_test_idx.npy: 467408 bytes
  team_train_idx.npy: 555752 bytes
  team_val_idx.npy: 554808 bytes
  y_test.npy: 58538 bytes

meta:
{
  "data_path": "/Users/sofya/Desktop/ВКР /data/ml_ozon_ounterfeit_train.csv",
  "random_state": 42,
  "split_strategy": "double GroupShuffleSplit by SellerID, test=0.30, val=0.50 of trainval",
  "sizes": {
    "train": 69453,
    "val": 69335,
    "test": 58410
  },
  "shares": {
    "train": 0.3521993123662512,
    "val": 0.35160092901550727,
    "test": 0.29619975861824155
  },
  "positive_rates": {
    "train": 0.0609908859228543,
    "val": 0.060186053219874525,
    "test": 0.07948981338811847
  },
  "n_team_features": 38,
  "team_drop_cols": [
    "id",
    "ItemID",
    "SellerID",
    "name_rus",
    "description",
    "brand_name",
    "text"
  ],
  "target": "resolution",
  "group": "SellerID",
  "df_n_rows": 197198,
  "df_n_cols": 45
}
